# Multi-dataset reasoning-graph evaluation — Colab T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arrafmousa/graph-of-thought/blob/master/notebooks/colab_run.ipynb)

This notebook evaluates whether token-level reasoning states can be merged safely.
Set **Runtime → Change runtime type → T4 GPU** before running it.

The configured experiment:
1. Loads five pinned Hugging Face math datasets: GSM8K, MATH-500, AIME 2025,
   SVAMP, and ASDiv.
2. Randomly samples exactly 20 questions from each dataset using recorded seeds.
3. Generates six complete reasoning chains per question with a frozen Llama 3.2 1B model.
4. Builds and saves a graph for every question under 3 merge heuristics × 5 thresholds.
5. Sends every unique accepted merge pair to the configured Azure OpenAI `gpt-5.1`
   **Batch deployment** for semantic-equivalence classification.
6. Computes same/different final-answer probabilities and a whole-graph quality score.
7. Downloads one archive containing all traces, hidden states, graph JSON, Azure batch
   records, metrics, and static HTML reports for offline inspection.

Before running, add two Colab secrets with notebook access enabled: `HF_TOKEN` and
`AZURE_OPENAI_API_KEY`. Never paste either secret into a notebook cell.

In [ ]:
# 1) Clone the repo fresh (restart-safe: always reset to /content and re-clone the latest master)
%cd /content
!rm -rf /content/graph-of-thought
# Public repo:
!git clone https://github.com/arrafmousa/graph-of-thought.git /content/graph-of-thought

# Private repo instead? Store a GitHub token as a Colab secret named GH_TOKEN, then:
# from google.colab import userdata
# tok = userdata.get('GH_TOKEN')
# !git clone https://{tok}@github.com/arrafmousa/graph-of-thought.git /content/graph-of-thought

%cd /content/graph-of-thought
# Print the exact running commit for reproducibility
!git log -1 --oneline

In [ ]:
# 2) Install runtime dependencies (Colab already provides CUDA-enabled torch)
!pip install -q -r requirements.txt
import torch
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 3) Load secrets into this ephemeral runtime without printing them
import os
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
azure_openai_key = userdata.get("AZURE_OPENAI_API_KEY")
if not hf_token:
    raise RuntimeError("Missing enabled Colab secret: HF_TOKEN")
if not azure_openai_key:
    raise RuntimeError("Missing enabled Colab secret: AZURE_OPENAI_API_KEY")
os.environ["AZURE_OPENAI_API_KEY"] = azure_openai_key
login(token=hf_token)
del hf_token, azure_openai_key
print("Hugging Face and Azure OpenAI credentials are configured.")

## Run the complete experiment

This is one reproducible run. Generation is checkpointed to per-question trace files;
all graph variants are written before the Azure Batch judge is submitted. Azure Batch
targets a 24-hour completion window, so the judging stage may remain active for a
substantial period even after GPU generation has finished.

In [ ]:
# 4) Generate traces, build every graph variant, submit the Azure Batch judge, and analyze results
import glob
import json
import os

# Pilot run: 5 questions per dataset. For the full/expensive run, switch to
# "configs/semantic_evaluation_pipeline/math/llama1b_five_dataset_azure_batch.json" (20 per dataset).
config_path = "configs/semantic_evaluation_pipeline/math/llama1b_five_dataset_pilot.json"
exit_code = get_ipython().system(f"python scripts/evaluate_merges.py --config {config_path}")
if exit_code != 0:
    raise RuntimeError(f"Semantic evaluation exited with code {exit_code}")

latest = max(
    glob.glob("output/*semantic-five-math*/"),
    key=os.path.getmtime,
).rstrip("/")
manifest = json.load(open(os.path.join(latest, "run_manifest.json"), encoding="utf-8"))
if manifest["status"] != "completed":
    raise RuntimeError(f"Run did not complete: {manifest.get('errors', [])}")
print("Completed run:", os.path.basename(latest))
print("Questions:", manifest["outputs"]["questions_sampled"])
print("Graphs:", manifest["outputs"]["graphs_built"])
print("Unique merge pairs judged:", manifest["outputs"]["unique_pairs_judged"])

## Inspect the result before downloading

The summary ranks every heuristic/threshold combination. The LLM judge sees only the
question and the two reasoning prefixes at the candidate merge; continuation outcomes
are measured afterward, independently. The full JSONL files remain available for
custom analysis.

In [ ]:
# 5) Preview the ranked configurations and static semantic report
from IPython.display import HTML, display

summary_path = os.path.join(latest, manifest["outputs"]["semantic_summary"])
summary = json.load(open(summary_path, encoding="utf-8"))
ranking = sorted(
    summary["configurations"],
    key=lambda row: row["mean_graph_quality_score"],
    reverse=True,
)
for row in ranking:
    print(
        f"{row['heuristic']} @ {row['threshold']}: "
        f"semantic={row['mean_semantic_score']:.3f}, "
        f"same-answer={row['same_final_answer_probability']}, "
        f"coverage={row['mean_coverage_proxy']:.3f}, "
        f"quality={row['mean_graph_quality_score']:.3f}"
    )
report_path = os.path.join(latest, manifest["outputs"]["semantic_report"])
display(HTML(open(report_path, encoding="utf-8").read()))

In [ ]:
# 6) Validate, package, and download the complete run
from pathlib import Path
import html
import subprocess
import sys
import zipfile

from google.colab import files

run_dir = Path(latest)
subprocess.run([sys.executable, "scripts/validate_run.py", str(run_dir)], check=True)
run_id = run_dir.name
bundle_name = f"{run_id}_offline_bundle"
archive_path = Path(f"{bundle_name}.zip")
index_html = f"""<!doctype html>
<html lang="en">
<head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Semantic merge evaluation</title></head>
<body style="font:16px/1.5 sans-serif;max-width:840px;margin:40px auto;padding:0 20px">
<h1>Semantic merge evaluation</h1>
<p>The complete reproducible run is under <code>output/{html.escape(run_id)}</code>.</p>
<ul>
<li><a href="output/{html.escape(run_id)}/{html.escape(manifest['outputs']['semantic_report'])}">Semantic evaluation report</a></li>
<li><a href="output/{html.escape(run_id)}/dashboard.html">Run dashboard</a></li>
<li><a href="output/{html.escape(run_id)}/{html.escape(manifest['outputs']['semantic_summary'])}">Machine-readable summary</a></li>
<li><a href="output/{html.escape(run_id)}/{html.escape(manifest['outputs']['graphs_index'])}">All inferred graph paths</a></li>
<li><a href="output/{html.escape(run_id)}/{html.escape(manifest['outputs']['merge_occurrences'])}">All merge judgments and continuation outcomes</a></li>
</ul>
<p>Extract the archive and open this file locally. All HTML reports are static.</p>
</body></html>
"""
readme = f"""Semantic merge evaluation bundle

Run: {run_id}

Open index.html first. To validate after copying the output directory into the repo:
python scripts/validate_run.py output/{run_id}

Key artifacts:
- all sampled source IDs and seeds: output/{run_id}/artifacts/traces/index.json
- all inferred graph JSON: output/{run_id}/artifacts/graphs/
- Azure Batch inputs/status/outputs: output/{run_id}/artifacts/evaluation/azure_batch/
- exhaustive judged merges: output/{run_id}/artifacts/evaluation/merge_occurrences.jsonl
- graph-level metrics: output/{run_id}/artifacts/evaluation/graph_quality.jsonl
- aggregate ranking: output/{run_id}/artifacts/evaluation/semantic_summary.json
"""
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    archive.writestr(f"{bundle_name}/index.html", index_html)
    archive.writestr(f"{bundle_name}/README.txt", readme)
    for artifact in run_dir.rglob("*"):
        if artifact.is_file():
            archive.write(artifact, arcname=f"{bundle_name}/{artifact.as_posix()}")

print(f"Bundle ready: {archive_path} ({archive_path.stat().st_size / (1024 ** 3):.2f} GiB)")
files.download(str(archive_path))

### Offline workflow

After downloading, extract the ZIP and open `index.html`. The archive contains all
100 sampled questions, complete generated chains and hidden states, every inferred
graph, every Azure Batch request/response, aggregate statistics, and selected visual
graph reports. You can change the seeds, sample sizes, merge sweep, or quality weights
in `configs/semantic_evaluation_pipeline/math/llama1b_five_dataset_azure_batch.json`
and rerun the notebook.